
假设我们有一个已经训练好的网络，它能够完成某个任务：
$$
H(x)=12
$$

现在我们想在这个网络中增加几层。

如果是普通网络，新增的网络层需要继续学习一个映射：
$$
x\longrightarrow H(x)
$$

假设新增的网络层没有带来任何有用的信息，我们希望它至少不要破坏原来的结果。

也就是说，希望它能够实现：
$$
H(x)=x
$$

但是普通网络需要通过权重和偏置学习这个恒等映射。

这并不是每次都容易做到的。

ResNet 将映射改成：
$$
\boxed{H(x)=x+F(x)}
$$

现在假设新增的网络层没有学到有用的东西。

我们只需要让：
$$
F(x)=0
$$

那么：
$$
H(x)=x+0=x
$$

原始信息就完整保留下来了。

**ResNet 真正厉害的地方不是它能识别哪些层没用，而是它提供了一种结构，让网络即使不需要改变原有特征，也能比较容易地保留这些特征。**

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=stride,padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1,)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=stride),
                nn.BatchNorm2d(out_channels)，
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        identity = self.shortcut(x)
        out = out + identity
        out = self.relu(out)
        return out